In [10]:
!pip install --upgrade transformers datasets accelerate torch tqdm

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/6a/6b/2f416568b3c4c91c96e5a365d164f8a4a4a88030aa8ab4644181fdadce97/transformers-4.57.3-py3-none-any.whl.metadata
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Obtaining dependency information for datasets from https://files.pythonhosted.org/packages/3b/5e/6f8d874366788ad5d549e9ba258037d974dda6e004843be1bda794571701/datasets-4.4.1-py3-none-any.whl.metadata
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Obtaining dependency information for tqdm from https://files.pythonhosted.org/packages/d0/30/dc54f88dd4a2b5dc8a0279bdd7270e735851848b762aeb1c1184ed1f6b14/tqdm-4.67.1-py3-none-any.whl.metadata
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Obtaining dependency information for tokenizers<=0.23.0,>=0.22.0 from https://files.pythonhosted.org/packages/b3/46/e33a8

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\HP\\AppData\\Roaming\\Python\\Python312\\site-packages\\~okenizers\\tokenizers.pyd'
Check the permissions.


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Cell 1 – Imports (no Trainer)

In [12]:
# %%
import os
# Tell transformers to ignore TensorFlow / Flax completely
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

# Install once (then comment out)
# !pip install --upgrade transformers datasets accelerate torch tqdm

import pandas as pd
from pathlib import Path
import numpy as np
import torch

from datasets import Dataset
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support,
    accuracy_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from torch.optim import AdamW          # <-- correct AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


# Cell 2 – Load combined CSV

In [13]:
# %%
ROOT = Path("..")
data_path = ROOT / "data" / "all_combined.csv"

df = pd.read_csv(data_path)
df["label"] = df["label"].astype(int)

print(df.shape)
df.head()


(84954, 4)


,text,label,source,script
0,Jo ata hy nea treqa batata hy kia lul mara how...,1,rut,roman
1,Lakh lanat ho terie shakal par... Aik chez dik...,1,rut,roman
2,Bakwaas bevqoof bna rahay ho logon ko,1,rut,roman
3,O lanti das bhi de,1,rut,roman
4,lun b gold ka hona chahiy tha,1,rut,roman


# Cell 3 – Train / val / test split

In [14]:
# %%
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42,
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)


Train: (67963, 4) Val: (8495, 4) Test: (8496, 4)


# Cell 4 – HuggingFace Datasets + tokenization

In [16]:
# %%
from datasets import config as datasets_config
import sys

# Make sure datasets does NOT try to use TensorFlow
datasets_config.TF_AVAILABLE = False
sys.modules.pop("tensorflow", None)  # remove any leftover stub

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_length = 128

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok   = val_ds.map(tokenize_batch, batched=True)
test_tok  = test_ds.map(tokenize_batch, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in cols_to_keep])
val_tok   = val_tok.remove_columns([c for c in val_tok.column_names if c not in cols_to_keep])
test_tok  = test_tok.remove_columns([c for c in test_tok.column_names if c not in cols_to_keep])

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

train_tok[0]


Map: 100%|██████████| 8496/8496 [00:01<00:00, 5386.12 examples/s]


{'label': tensor(0),
 'input_ids': tensor([   101,  11038,  10104,  22394,  19180,  33753,  10263,  10879,  29956,
          49537,  10116,  10730,  24948,  13321,  35102,  25085,    170,  11924,
          35595,  10116,  11162,  10116,  10730,  74675,  10126,  12438,  62137,
          87147,  33551,  13998,  10106,  14786,  14100,  82608,  37878,  28248,
          11030,  10126,  11346,  10174,  16028,  11669,  20316,  30562,  22953,
          13998,  11163,  10339, 100900,  10112,  40565,  10927,  11163,  10339,
          10835,    146,  11703,  26680,  10354,    154,  41971,  10129,  10141,
          10201,  10679,  45365,    102,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,   

# Cell 5 – Dataloaders

In [17]:
# %%
batch_size = 16

train_loader = DataLoader(train_tok, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_tok, batch_size=32)
test_loader  = DataLoader(test_tok, batch_size=32)


# Cell 6 – Model + optimizer + scheduler

In [18]:
# %%
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)


Using device: cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Cell 7 – Helper: eval function

In [21]:
# %%
def eval_loop(data_loader):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in data_loader:
            # Separate labels and inputs
            labels = batch["label"].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}

            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    avg_loss = total_loss / len(data_loader)

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }, all_labels, all_preds


# Cell 8 – Training loop

In [22]:
# %%
for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for i, batch in enumerate(pbar, start=1):
        # Separate labels and inputs
        labels = batch["label"].to(device)
        inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_train_loss += loss.item()
        pbar.set_postfix({"train_loss": total_train_loss / i})

    train_loss = total_train_loss / len(train_loader)
    val_metrics, _, _ = eval_loop(val_loader)

    print(f"\nEpoch {epoch+1}:")
    print(f"  Train loss: {train_loss:.4f}")
    print(
        f"  Val loss: {val_metrics['loss']:.4f} | "
        f"Acc: {val_metrics['accuracy']:.3f} | "
        f"Prec: {val_metrics['precision']:.3f} | "
        f"Rec: {val_metrics['recall']:.3f} | "
        f"F1: {val_metrics['f1']:.3f}"
    )


Epoch 1/3:   1%|          | 24/4248 [04:31<13:16:36, 11.32s/it, train_loss=0.567]


KeyboardInterrupt: 

# Cell 9 – Final test evaluation + classification report

In [ ]:
# %%
test_metrics, y_true, y_pred = eval_loop(test_loader)

print("Test metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nClassification report (test set):\n")
print(classification_report(y_true, y_pred, digits=3))


# Cell 10 – Save model + tokenizer

In [ ]:
# %%
save_dir = ROOT / "models" / "mbert_toxic_final"
save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved to:", save_dir)


In [1]:
# %%
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ROOT = Path("..")
model_dir = ROOT / "models" / "mbert_toxic_final"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_proba_mbert(text: str) -> float:
    enc = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1)
        return probs[0, 1].item()

def predict_label_mbert(text: str, threshold: float = 0.5) -> int:
    return int(predict_proba_mbert(text) >= threshold)

print(predict_proba_mbert("tum bohat gande insan ho"))
print(predict_proba_mbert("bahut shukriya, bohat acha content tha"))


C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.8078837394714355
0.0034208500292152166


In [2]:
# %%
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ROOT = Path("..")
model_dir = ROOT / "models" / "mbert_toxic_final"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_proba_mbert(text: str) -> float:
    """Return P(toxic=1) for a single text."""
    enc = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1)
        p_toxic = probs[0, 1].item()
    return p_toxic

def predict_label_mbert(text: str, threshold: float = 0.5) -> int:
    return int(predict_proba_mbert(text) >= threshold)

# quick sanity check
print(predict_proba_mbert("tum bohat gande insan ho"))
print(predict_proba_mbert("bahut shukriya, bohat acha content tha"))


0.8078837394714355
0.0034208500292152166
